In [ ]:
/**
 * DATABASE TRANSACTIONS — Complete Guide
 * 1. WHAT IS A TRANSACTION? ............. Simple explanation + real-life example
 * 2. WHY DO WE NEED TRANSACTIONS? ....... Real problems it solves
 * 3. ACID PROPERTIES .................... 4 important rules
 * 4. TRANSACTION ANOMALIES .............. What can go wrong
 * 5. ISOLATION LEVELS ................... How to control safety vs speed
 * 6. PRACTICAL EXAMPLES ................. PostgreSQL code
 * 7. GOLDEN RULES ....................... Key principles to remember
 * 8. PRACTICE EXERCISES ................. Try yourself
 * 9. QUICK REFERENCE CARD ............... Commands at a glance
 */

In [ ]:
/**
 * Part-1: WHAT IS A TRANSACTION?
 */

In [ ]:
/**
 * REAL-LIFE EXAMPLE — Money Transfer
 * ────────────────────────────────────
 *
 *   Socho tum apne dost ko ₹100 bhejte ho.
 *   Isme 2 steps hain:
 *     Step 1: Tumhare account se ₹100 katao
 *     Step 2: Dost ke account mein ₹100 daalo
 *
 *   PROBLEM: Agar Step 1 ho gaya aur Step 2 fail ho gaya?
 *     → Tumhara ₹100 cut ho gaya
 *     → Dost ko nahi mila
 *     → ₹100 hawa mein gaya!
 *
 *   SOLUTION: TRANSACTION use karo
 *     → Dono steps ek saath hote hain
 *     → Ya DONO succeed karo, ya KUCH BHI nahi hoga
 *     → Paisa safe!
 *
 * DEFINITION:
 *   Transaction = Database operations ka ek group jo ek unit ki tarah chalta hai.
 *   Ya SAARI operations succeed hongi, ya KUCH BHI nahi hoga.
 *
 * KEY CHARACTERISTICS:
 *   1. Multiple steps hote hain (INSERT, UPDATE, DELETE)
 *   2. Ek unit ki tarah treat hota hai — break nahi kar sakte
 *   3. Data read ya write karta hai
 *   4. Ya pura complete hota hai (COMMIT) ya pura fail hota hai (ROLLBACK)
 */

In [ ]:
/**
 * Part-2: WHY DO WE NEED TRANSACTIONS? — Real Problems
 */

In [ ]:
/**
 * PROBLEM 1: PARTIAL UPDATES — System crash beech mein
 * ──────────────────────────────────────────────────────
 *
 *   Bank transfer ke beech mein system crash ho gaya.
 *   Step 1 complete hua, Step 2 nahi hua.
 *   Paisa gaya!
 *
 *   Example:
 *     Step 1: Account A se ₹100 nikalo  ✓ (ho gaya)
 *     Step 2: Account B mein ₹100 daalo ✗ (CRASH!)
 *
 *   Result: Account A mein kam paisa, Account B mein kuch nahi.
 *   Solution: Transaction Step 1 ko automatically UNDO kar deta hai agar Step 2 fail ho.
 *
 * PROBLEM 2: CONCURRENCY ISSUES — Ek saath 2 log
 * ──────────────────────────────────────────────────
 *
 *   Flight ki last seat book karne ki koshish 2 log ek saath kar rahe hain.
 *
 *   Person A: Available seats check kiya → 1 seat left
 *   Person B: Available seats check kiya → 1 seat left
 *   Person A: Seat book ki ✓
 *   Person B: Seat book ki ✓ (lekin seat toh thi hi nahi!)
 *
 *   Result: DOUBLE BOOKING! Ek hi seat 2 logon ko.
 *   Solution: Transaction isolation ye prevent karta hai.
 *
 * PROBLEM 3: DATA LOSS — System failure ke baad changes gayab
 * ─────────────────────────────────────────────────────────────
 *
 *   Customer ne ₹5000 ka order place kiya, DB ne save kiya, turant crash ho gaya.
 *   Restart ke baad: ORDER GONE!
 *
 *   Customer sochta hai order place hua, system mein kuch nahi.
 *   Solution: Durability ensure karta hai ki committed changes permanent hain.
 */

In [ ]:
/**
 * Part-3: ACID PROPERTIES — 4 Important Rules
 */

In [ ]:
/**
 * ACID kya hai?
 * ─────────────
 *
 *   ACID = 4 rules jo har transaction ko follow karne chahiye.
 *   Ye rules ensure karte hain ki data safe aur correct rahe.
 *
 *   A = Atomicity    (All or Nothing)
 *   C = Consistency  (Rules always follow honge)
 *   I = Isolation    (Koi interference nahi)
 *   D = Durability   (Permanent save)
 */

/**
 * RULE 1: ATOMICITY — All or Nothing
 * ────────────────────────────────────
 *
 *   Transaction ek atom ki tarah hai — tod nahi sakte.
 *   Ya SAARE steps hote hain, ya EK BHI nahi.
 *   Problem solved: Partial updates.
 *
 *   Example: Money transfer
 *
 *   BEGIN TRANSACTION;
 *       UPDATE account SET balance = balance - 100 WHERE user_id = 1;
 *       UPDATE account SET balance = balance + 100 WHERE user_id = 2;
 *   COMMIT;
 *
 *   Agar second UPDATE fail ho → first UPDATE automatically UNDO ho jaata hai.
 *   Paisa safe!
 *
 *   Without Atomicity:
 *     Step 1: ₹100 nikalo ✓
 *     Step 2: ₹100 daalo  ✗ (FAILED)
 *     Result: Paisa gaya!  💔
 *
 *   With Atomicity:
 *     Step 1: ₹100 nikalo ✓
 *     Step 2: ₹100 daalo  ✗ (FAILED)
 *     Result: Step 1 UNDO, paisa tumhare account mein safe ✓
 *
 * RULE 2: CONSISTENCY — Data Integrity
 * ──────────────────────────────────────
 *
 *   Database rules HAMESHA follow honge.
 *   Koi bhi invalid data kabhi save nahi hoga.
 *   Problem solved: Data integrity.
 *
 *   Example: Bank account negative nahi ho sakta.
 *   Rule: balance >= 0
 *
 *   BEGIN TRANSACTION;
 *       UPDATE account SET balance = balance - 500 WHERE user_id = 1;
 *   COMMIT;
 *
 *   Agar user ke paas sirf ₹100 hai → balance -₹400 ho jaata (INVALID!)
 *   Transaction automatically FAIL ho jaata, koi change nahi hota.
 *
 *   Other rules ke examples:
 *     → Age negative nahi ho sakti
 *     → Email unique hona chahiye
 *     → Order status valid hona chahiye (PLACED, PAID, etc.)
 *
 * RULE 3: ISOLATION — No Interference
 * ──────────────────────────────────────
 *
 *   Multiple transactions ek saath chalte hain — ek doosre ko disturb nahi karte.
 *   Har transaction sochta hai wo akela chal raha hai.
 *   Problem solved: Concurrency issues (double booking, etc.)
 *
 *   Example: Flight seat booking
 *
 *   Transaction A (Person 1): Seat 5A book karna chahta hai
 *   Transaction B (Person 2): Seat 5A book karna chahta hai
 *
 *   With Isolation:
 *     Transaction A pehle chalta hai → seat 5A book karta hai
 *     Transaction B wait karta hai → dekhta hai seat 5A le li gayi → doosri seat leta hai
 *
 *   Without Isolation:
 *     Dono ek hi seat book kar lete hain → DOUBLE BOOKING! ✗
 *
 * RULE 4: DURABILITY — Permanent Save
 * ─────────────────────────────────────
 *
 *   Ek baar transaction commit ho gaya → changes PERMANENT hain.
 *   System crash bhi ho jaaye → data safe hai.
 *   Problem solved: Data loss after crash.
 *
 *   Example: Online order
 *     Customer ne ₹5000 ka order diya.
 *     Transaction commit (disk pe save).
 *     Turant system crash.
 *     Restart ke baad: ORDER ABHI BHI HAI ✓
 *
 *   Without Durability:
 *     Order sirf memory mein tha.
 *     System crash → Order gaya ✗
 */

/**
 * ACID SUMMARY TABLE:
 * ┌─────────────────┬───────────────────────────┬──────────────────────────────┐
 * │ Property        │ Problem Solved            │ Simple Meaning               │
 * ├─────────────────┼───────────────────────────┼──────────────────────────────┤
 * │ Atomicity       │ Partial updates           │ All or nothing               │
 * │ Consistency     │ Invalid data              │ Follow all rules             │
 * │ Isolation       │ Concurrency conflicts     │ No interference              │
 * │ Durability      │ Data loss after crash     │ Permanent save               │
 * └─────────────────┴───────────────────────────┴──────────────────────────────┘
 */

In [ ]:
/**
 * Part-4: TRANSACTION ANOMALIES — What Can Go Wrong
 */

In [ ]:
/**
 * Anomaly kya hota hai?
 * ──────────────────────
 *
 *   Anomaly = kuch galat jo ho jaata hai.
 *   Jab transactions proper isolation ke bina ek saath chalte hain,
 *   3 tarah ki problems ho sakti hain.
 */

/**
 * ANOMALY 1: DIRTY READ — Unsaved data padhna
 * ─────────────────────────────────────────────
 *
 *   Aisa data padhna jo abhi COMMIT nahi hua — sirf temporary change hai.
 *   Agar wo transaction rollback kar de → tumne galat data padha tha.
 *
 *   Example:
 *     Transaction T1 (Bank):
 *       UPDATE account SET balance = balance - 100 WHERE user_id = 1;
 *       -- Balance ab ₹400 hai (lekin save nahi hua!)
 *
 *     Transaction T2 (Customer balance check):
 *       SELECT balance FROM account WHERE user_id = 1;
 *       -- ₹400 dikhta hai (but ye final nahi!)
 *
 *     Transaction T1 (Bank):
 *       ROLLBACK;   -- kuch galat hua, cancel karo
 *       -- Balance wapas ₹500 ho gaya
 *
 *     PROBLEM: T2 ne ₹400 dekha, lekin actual balance ₹500 hai!
 *              T2 ne GALAT data ke saath kaam kiya.
 *
 *   Visual timeline:
 *     T1: ₹400 pe update kiya (save nahi)
 *     T2: ₹400 padha          ← DIRTY READ!
 *     T1: ROLLBACK (₹500 wapas)
 *     T2: Galat data use kiya!
 *
 * ANOMALY 2: NON-REPEATABLE READ — Ek hi data 2 baar padho, alag result
 * ─────────────────────────────────────────────────────────────────────
 *
 *   Ek hi data 2 baar padha, dono baar alag value aayi.
 *   Beech mein kisi doosre transaction ne change kar diya.
 *
 *   Example:
 *     Transaction T1 (Report generator):
 *       SELECT balance FROM account WHERE user_id = 1;
 *       -- ₹500 mila
 *
 *     Transaction T2 (Bank deposit):
 *       UPDATE account SET balance = balance + 200 WHERE user_id = 1;
 *       COMMIT;   -- Balance ab ₹700 hai
 *
 *     Transaction T1 (Report generator):
 *       SELECT balance FROM account WHERE user_id = 1;
 *       -- ₹700 mila (pehle se alag!)
 *
 *     PROBLEM: Same query ne alag result diya!
 *              Report inconsistent ban gayi.
 *
 *   Visual timeline:
 *     T1: ₹500 padha
 *     T2: ₹700 kiya aur COMMIT
 *     T1: ₹700 padha ← NON-REPEATABLE READ!
 *     T1: Ek hi data ke 2 alag values mili
 *
 * ANOMALY 3: PHANTOM READ — Rows ki ginti badal gayi
 * ────────────────────────────────────────────────────
 *
 *   Same query run karo — rows ki ginti badal jaati hai.
 *   Nayi rows "ghost ki tarah" appear ho jaati hain.
 *
 *   Example:
 *     Transaction T1 (Report generator):
 *       SELECT COUNT(*) FROM users WHERE balance > 1000;
 *       -- 2 rows mili
 *
 *     Transaction T2 (New user registration):
 *       INSERT INTO users VALUES (100, 'New User', 1500);
 *       COMMIT;   -- Naya user add hua
 *
 *     Transaction T1 (Report generator):
 *       SELECT COUNT(*) FROM users WHERE balance > 1000;
 *       -- 3 rows mili (pehle se alag!)
 *
 *     PROBLEM: Nayi rows phantom ki tarah appear hui!
 *              Report unexpected change ho gayi.
 *
 *   Visual timeline:
 *     T1: Count = 2 rows
 *     T2: Naya row INSERT kiya aur COMMIT
 *     T1: Count = 3 rows ← PHANTOM READ!
 *     T1: Alag rows ki ginti mili
 */

In [ ]:
/**
 * Part-5: ISOLATION LEVELS — How to Control Safety vs Speed
 */

In [ ]:
/**
 * Isolation Level kya hota hai?
 * ──────────────────────────────
 *
 *   Isolation level = tum kitna strict rehna chahte ho.
 *   Zyada isolation = zyada safe, lekin slow.
 *   Kam isolation = kam safe, lekin fast.
 *
 *   4 levels hain — least safe se most safe:
 *     1. Read Uncommitted
 *     2. Read Committed
 *     3. Repeatable Read
 *     4. Serializable
 */

/**
 * LEVEL 1: READ UNCOMMITTED — Sabse kam safe
 * ──────────────────────────────────────────────
 *
 *   Uncommitted data bhi padh sakte ho — dirty reads allowed.
 *   Production mein KABHI USE MAT KARO — bahut risky.
 *
 *   Anomalies possible:
 *     ✓ Dirty Reads          (ho sakta hai)
 *     ✓ Non-Repeatable Reads (ho sakta hai)
 *     ✓ Phantom Reads        (ho sakta hai)
 *
 *   Example of the problem:
 *     T1: balance ₹400 kiya (commit nahi)
 *     T2: ₹400 padha  (T1 rollback kar de toh?)
 *     T1: ROLLBACK → balance ₹500 wapas
 *     T2: Galat data use kiya!
 *
 * LEVEL 2: READ COMMITTED — Default in PostgreSQL
 * ──────────────────────────────────────────────────
 *
 *   Sirf committed data padh sakte ho.
 *   Dirty reads nahi hote.
 *   Ye sabse common choice hai — safety aur speed ka balance.
 *
 *   Anomalies possible:
 *     ✗ Dirty Reads          (nahi hoga)
 *     ✓ Non-Repeatable Reads (ho sakta hai)
 *     ✓ Phantom Reads        (ho sakta hai)
 *
 *   Example:
 *     T1: ₹400 kiya aur COMMIT
 *     T2: ₹400 padha — SAFE (T1 commit kar chuka hai)
 *
 *     BUT agar T1 same data 2 baar padhe:
 *       T1: Pehle ₹500 padha
 *       T2: ₹700 kiya aur COMMIT
 *       T1: Doosri baar ₹700 padha → ALAG! (non-repeatable read)
 *
 * LEVEL 3: REPEATABLE READ — Zyada safe
 * ──────────────────────────────────────
 *
 *   Ek baar jo data padha, wo same rahega transaction ke end tak.
 *   Reports aur analytics ke liye useful.
 *
 *   Anomalies possible:
 *     ✗ Dirty Reads          (nahi hoga)
 *     ✗ Non-Repeatable Reads (nahi hoga)
 *     ✓ Phantom Reads        (PostgreSQL mein bhi prevent hota hai)
 *
 *   Example:
 *     T1: Pehle ₹500 padha
 *     T2: ₹700 karna chahta hai — BLOCKED (T1 ke khatam hone tak ruko)
 *     T1: Doosri baar ₹500 padha — SAME! (correct)
 *     T1: COMMIT
 *     T2: Ab update allowed
 *
 *     BUT nayi rows abhi bhi aa sakti hain:
 *       T1: Count = 2 rows
 *       T2: INSERT kiya aur COMMIT
 *       T1: Count = 3 rows (phantom read)
 *
 * LEVEL 4: SERIALIZABLE — Sabse safe
 * ─────────────────────────────────────
 *
 *   Transactions ek ke baad ek chale — koi parallel nahi.
 *   Banking, financial systems ke liye use karo.
 *   Safest lekin slowest.
 *
 *   Anomalies possible:
 *     ✗ Dirty Reads          (nahi hoga)
 *     ✗ Non-Repeatable Reads (nahi hoga)
 *     ✗ Phantom Reads        (nahi hoga)
 *
 *   Example:
 *     T1: ₹500 padha
 *     T2: ₹700 karna chahta hai — BLOCKED
 *     T1: Count = 2 rows
 *     T2: INSERT karna chahta hai — BLOCKED
 *     T1: COMMIT
 *     T2: Ab chalne diya
 *     Sab consistent!
 */

/**
 * ISOLATION LEVELS SUMMARY TABLE:
 * ┌─────────────────────┬─────────────┬──────────────────────┬───────────────┐
 * │ Isolation Level     │ Dirty Reads │ Non-Repeatable Reads │ Phantom Reads │
 * ├─────────────────────┼─────────────┼──────────────────────┼───────────────┤
 * │ Read Uncommitted    │ YES (bad)   │ YES (bad)            │ YES (bad)     │
 * │ Read Committed      │ NO  (good)  │ YES (bad)            │ YES (bad)     │
 * │ Repeatable Read     │ NO  (good)  │ NO  (good)           │ YES (bad)     │
 * │ Serializable        │ NO  (good)  │ NO  (good)           │ NO  (good)    │
 * └─────────────────────┴─────────────┴──────────────────────┴───────────────┘
 *
 * YES = Problem ho sakta hai (BAD)
 * NO  = Problem nahi hoga    (GOOD)
 *
 * Performance (fast → slow):
 *   Read Uncommitted → Read Committed → Repeatable Read → Serializable
 *   (fastest)                                           (slowest)
 *
 * Safety (low → high):
 *   Read Uncommitted → Read Committed → Repeatable Read → Serializable
 *   (least safe)                                       (most safe)
 */

In [ ]:
/**
 * Part-6: PRACTICAL EXAMPLES — PostgreSQL Code
 */

In [ ]:
/**
 * SETUP: Sample bank accounts table
 * ───────────────────────────────────
 *
 *   Pehle ek bank accounts table banate hain practice ke liye.
 */


-- Table create karo
CREATE TABLE accounts (
    account_id SERIAL PRIMARY KEY,
    user_name  VARCHAR(100),
    balance    DECIMAL(10,2) NOT NULL CHECK (balance >= 0)
);

-- Sample data daalo
INSERT INTO accounts (user_name, balance) VALUES
    ('Raj',   1000.00),
    ('Priya',  500.00),
    ('Amit',  2000.00);

In [ ]:
/**
 * EXAMPLE 1: Basic Transaction — BEGIN, COMMIT, ROLLBACK
 * ────────────────────────────────────────────────────────
 *
 *   Raj se Priya ko ₹200 transfer karo.
 *   BEGIN → steps karo → COMMIT agar sab sahi, ROLLBACK agar kuch galat.
 */


-- Transaction start karo
BEGIN;

-- Step 1: Raj se ₹200 lo
UPDATE accounts SET balance = balance - 200 WHERE user_name = 'Raj';

-- Step 2: Priya ko ₹200 do
UPDATE accounts SET balance = balance + 200 WHERE user_name = 'Priya';

-- Check karo sab sahi hai?
SELECT * FROM accounts;

-- Sab sahi lag raha hai → save karo
COMMIT;

-- Kuch galat hua → ROLLBACK;

In [ ]:
/**
 * EXAMPLE 2: Transaction with Error Handling
 * ───────────────────────────────────────────
 *
 *   Safe money transfer — pehle balance check karo, phir transfer karo.
 *   EXCEPTION block se error handle karo aur ROLLBACK karo.
 */


DO $$
DECLARE
    v_raj_balance DECIMAL;
BEGIN
    BEGIN
        -- Raj ka balance check karo
        SELECT balance INTO v_raj_balance
        FROM accounts WHERE user_name = 'Raj';

        -- Agar balance kam hai toh error
        IF v_raj_balance < 500 THEN
            RAISE EXCEPTION 'Insufficient balance! Raj ke paas sirf % hai', v_raj_balance;
        END IF;

        -- Transfer karo
        UPDATE accounts SET balance = balance - 500 WHERE user_name = 'Raj';
        UPDATE accounts SET balance = balance + 500 WHERE user_name = 'Priya';

        COMMIT;
        RAISE NOTICE 'Transfer successful!';

    EXCEPTION
        WHEN OTHERS THEN
            -- Kuch bhi galat hua → sab undo karo
            ROLLBACK;
            RAISE NOTICE 'Transfer failed: %', SQLERRM;
    END;
END;
$$;

In [ ]:
/**
 * EXAMPLE 3: SAVEPOINT — Partial Rollback
 * ─────────────────────────────────────────
 *
 *   SAVEPOINT = transaction ke andar ek bookmark.
 *   Pura transaction rollback nahi karna, sirf ek part undo karna.
 *   Jaise ek game mein checkpoint — wahan se dobara shuru kar sakte ho.
 */


BEGIN;

UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Raj';

-- Savepoint banao
SAVEPOINT after_raj_debit;

UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Amit';

-- Oops! Amit ke paas enough paisa nahi — sirf Amit wala undo karo
ROLLBACK TO SAVEPOINT after_raj_debit;

-- Raj wala update abhi bhi hai!
-- Ab kuch aur try karo
UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Priya';

-- Sab sahi → commit karo
COMMIT;

In [ ]:
/**
 * EXAMPLE 4: Isolation Level Check aur Change
 * ─────────────────────────────────────────────
 *
 *   Current isolation level dekho aur zaroorat ke hisaab se change karo.
 */


-- Current isolation level check karo
SHOW transaction_isolation;

-- Session ke liye change karo
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;

-- Specific level ke saath transaction
BEGIN;
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;
-- Queries yahan...
COMMIT;

In [ ]:
/**
 * EXAMPLE 5: Dirty Read Prevention — 2 Terminals
 * ─────────────────────────────────────────────────
 *
 *   READ COMMITTED mein dirty reads nahi hote.
 *   Terminal 2 tabhi naya data dekhega jab Terminal 1 commit kare.
 */


-- TERMINAL 1 (Transaction 1)
BEGIN;
UPDATE accounts SET balance = balance - 300 WHERE user_name = 'Raj';
-- Commit ABHI MAT KARO

-- TERMINAL 2 (Transaction 2)
-- READ COMMITTED ke saath, ye T1 ke commit ka wait karega
BEGIN;
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
SELECT * FROM accounts WHERE user_name = 'Raj';
-- Purana balance dikhega (1000) jab tak T1 commit na kare
COMMIT;

-- Wapas TERMINAL 1
COMMIT;

In [ ]:
/**
 * EXAMPLE 6: Phantom Read Demo
 * ──────────────────────────────
 *
 *   REPEATABLE READ mein PostgreSQL phantom reads bhi prevent karta hai.
 *   Terminal 1 mein count wahi rahega chahe Terminal 2 kuch insert kare.
 */


-- TERMINAL 1 (Transaction 1)
BEGIN;
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
SELECT COUNT(*) FROM accounts WHERE balance > 500;
-- Maano 2 rows aaye

-- TERMINAL 2 (Transaction 2)
BEGIN;
INSERT INTO accounts (user_name, balance) VALUES ('Neha', 1000);
COMMIT;

-- Wapas TERMINAL 1
SELECT COUNT(*) FROM accounts WHERE balance > 500;
-- REPEATABLE READ mein abhi bhi 2 aayenge (PostgreSQL phantom prevent karta hai)
COMMIT;

In [ ]:
/**
 * EXAMPLE 7: Real E-commerce Order Transaction
 * ──────────────────────────────────────────────
 *
 *   Ek poora order process karo ek hi transaction mein:
 *   product price check → order create → items add → payment record.
 *   Koi bhi step fail ho → sab kuch rollback.
 */


CREATE OR REPLACE FUNCTION process_order(
    p_user_id    BIGINT,
    p_product_id BIGINT,
    p_quantity   INT
)
RETURNS TEXT
LANGUAGE plpgsql
AS $$
DECLARE
    v_product_price DECIMAL;
    v_total_amount  DECIMAL;
BEGIN
    BEGIN
        -- Product ka price lo
        SELECT price INTO v_product_price
        FROM products WHERE product_id = p_product_id;

        -- Total calculate karo
        v_total_amount := v_product_price * p_quantity;

        -- Order create karo
        INSERT INTO orders (order_id, user_id, status, created_at, total_amount)
        VALUES (nextval('order_seq'), p_user_id, 'PLACED', NOW(), v_total_amount);

        -- Order item add karo
        INSERT INTO order_items (order_id, product_id, qty, unit_price)
        VALUES (currval('order_seq'), p_product_id, p_quantity, v_product_price);

        -- Payment record banao
        INSERT INTO payments (order_id, currency, amount, status, created_at)
        VALUES (currval('order_seq'), 'INR', v_total_amount, 'SUCCESS', NOW());

        -- Sab step sahi hue → save karo
        COMMIT;
        RETURN 'Order placed successfully! Order ID: ' || currval('order_seq');

    EXCEPTION
        WHEN OTHERS THEN
            -- Koi bhi step fail hua → sab undo karo
            ROLLBACK;
            RETURN 'Order failed: ' || SQLERRM;
    END;
END;
$$;

In [ ]:
/**
 * EXAMPLE 8: Long-Running Transactions Monitor Karo
 * ────────────────────────────────────────────────────
 *
 *   Khuli padi transactions resources lock kar ke rakhti hain.
 *   Ye query dhoondhti hai ki kaun si transactions bahut der se chal rahi hain.
 */


SELECT
    pid,
    usename,
    state,
    query,
    now() - xact_start AS transaction_duration
FROM pg_stat_activity
WHERE state = 'idle in transaction'
ORDER BY transaction_duration DESC;

In [ ]:
/**
 * Part-7: GOLDEN RULES — Key Principles to Remember
 */

In [ ]:
/**
 * GOLDEN RULES — Database Transactions ke Core Principles
 * ─────────────────────────────────────────────────────────
 *
 *   Ye 8 rules yaad rakho — production mein transactions theek se use karne ke liye.
 *
 *  1. ✅ MULTIPLE RELATED OPERATIONS = HAMESHA TRANSACTION USE KARO
 *        Bank transfer, order processing, inventory update —
 *        agar 1 se zyada INSERT/UPDATE hai → transaction use karo.
 *
 *  2. ✅ HAMESHA ERROR HANDLE KARO (EXCEPTION BLOCK)
 *        Ye mat socho ki sab kuch work karega.
 *        Hamesha ROLLBACK plan rakho.
 *
 *  3. ✅ TRANSACTIONS SHORT RAKHO
 *        Lambi transactions tables lock kar ke slow kar deti hain.
 *        Sirf zaroori kaam transaction ke andar karo.
 *
 *  4. ✅ EDGE CASES TEST KARO
 *        Kya ho agar balance insufficient ho?
 *        Kya ho agar product exist na kare?
 *        Kya ho agar database crash ho jaaye?
 *
 *  5. ✅ ISOLATION LEVEL SOCH SAMAJH KE CHOOSE KARO
 *        READ COMMITTED → most cases ke liye sahi (PostgreSQL default)
 *        SERIALIZABLE  → financial/medical data ke liye
 *        READ UNCOMMITTED → almost kabhi use mat karo
 *
 *  6. ✅ HAMESHA COMMIT YA ROLLBACK KARO
 *        Transaction kabhi open mat chhodo.
 *        Open transactions resources lock kar ke rakhte hain.
 *
 *  7. ✅ PARTIAL ROLLBACK KE LIYE SAVEPOINT USE KARO
 *        Jab sirf transaction ka kuch hissa undo karna ho.
 *        Batch operations mein useful.
 *
 *  8. ✅ LONG-RUNNING TRANSACTIONS MONITOR KARO
 *        pg_stat_activity se dhundho.
 *        Lambi transactions performance hurt karti hain.
 */

In [ ]:
/**
 * Part-8: PRACTICE EXERCISES — Try Yourself
 */

In [ ]:
/**
 * EXERCISE 1: Simple Money Transfer
 * ────────────────────────────────────
 * Question: Amit se Priya ko ₹500 transfer karo.
 *           BEGIN, UPDATE, COMMIT use karo.
*/

BEGIN;
UPDATE accounts SET balance = balance - 500 WHERE user_name = 'Amit';
UPDATE accounts SET balance = balance + 500 WHERE user_name = 'Priya';
COMMIT;

In [ ]:
/**
 * EXERCISE 2: Insufficient Balance Check
 * ────────────────────────────────────────
 * Question: Raj ke paas ₹1000 hai, ₹3000 transfer karne ki koshish karo.
 *           Fail hona chahiye aur rollback hona chahiye.
*/

DO $$
DECLARE v_balance DECIMAL;
BEGIN
    SELECT balance INTO v_balance FROM accounts WHERE user_name = 'Raj';
    IF v_balance < 3000 THEN
        RAISE EXCEPTION 'Insufficient balance';
    END IF;
    BEGIN
        UPDATE accounts SET balance = balance - 3000 WHERE user_name = 'Raj';
        UPDATE accounts SET balance = balance + 3000 WHERE user_name = 'Priya';
        COMMIT;
    EXCEPTION WHEN OTHERS THEN
        ROLLBACK;
        RAISE NOTICE 'Transfer failed';
    END;
END;
$$;

In [ ]:
/**
 * EXERCISE 3: Current Isolation Level Check Karo
 * ────────────────────────────────────────────────
 * Question: PostgreSQL mein current isolation level kaise dekhte hain?
*/

SHOW transaction_isolation;

In [ ]:
/**
 * EXERCISE 4: Isolation Level Set Karo
 * ──────────────────────────────────────
 * Question: Current session ka isolation level SERIALIZABLE set karo.
*/

SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;

In [ ]:
/**
 * EXERCISE 5: SAVEPOINT Use Karo
 * ────────────────────────────────
 * Question: Teen updates karo, sirf last wala undo karo.
*/

BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Raj';
SAVEPOINT after_first;
UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Amit';
SAVEPOINT after_second;
UPDATE accounts SET balance = balance - 100 WHERE user_name = 'Priya';
-- Oops! Sirf last wala undo karo
ROLLBACK TO SAVEPOINT after_second;
COMMIT;

In [ ]:
/**
 * Part-9: QUICK REFERENCE CARD
 */

In [ ]:
/**
 * COMMANDS AT A GLANCE:
 * ┌───────────────────────────────┬────────────────────────────────────────┐
 * │ Command                       │ Kya karta hai                          │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ BEGIN;                        │ Transaction start karo                 │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ COMMIT;                       │ Sab changes save karo                  │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ ROLLBACK;                     │ Sab changes undo karo                  │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ SAVEPOINT name;               │ Rollback ke liye ek point mark karo    │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ ROLLBACK TO SAVEPOINT name;   │ Savepoint tak undo karo                │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ RELEASE SAVEPOINT name;       │ Savepoint hata do                      │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ SHOW transaction_isolation;   │ Current isolation level dekho          │
 * ├───────────────────────────────┼────────────────────────────────────────┤
 * │ SET TRANSACTION ISOLATION     │ Isolation level change karo            │
 * │ LEVEL <level>;                │                                        │
 * └───────────────────────────────┴────────────────────────────────────────┘
*/

In [ ]:
/**
 * ISOLATION LEVEL QUICK GUIDE:
 * ┌──────────────────────┬───────────────────────────────────────────────┐
 * │ Level                │ Kab use karo                                  │
 * ├──────────────────────┼───────────────────────────────────────────────┤
 * │ Read Uncommitted     │ Kabhi mat use karo (too risky)                │
 * │ Read Committed       │ Default, most cases ke liye sahi              │
 * │ Repeatable Read      │ Reports, analytics jahan consistent reads ho  │
 * │ Serializable         │ Banking, financial — bilkul galati nahi chalti│
 * └──────────────────────┴───────────────────────────────────────────────┘
 *
 * Performance: Read Uncommitted (fastest) → Serializable (slowest)
 * Safety:      Read Uncommitted (least)   → Serializable (most)
 */

In [ ]:
/**
 * FINAL SUMMARY — Ye sab tumne seekha
 */

In [ ]:
/**
 * 1. TRANSACTION kya hai?
 *    = Operations ka ek group jo saath succeed karein ya saath fail karein.
 *      Tumhare data ka bodyguard — crashes, conflicts, corruption se protect karta hai.
 *
 * 2. ACID ke 4 rules:
 *    → Atomicity   : All or nothing — ya sab, ya kuch nahi
 *    → Consistency : Rules always follow honge — invalid data kabhi save nahi
 *    → Isolation   : No interference — har transaction akela chalta hai
 *    → Durability  : Permanent save — commit ke baad crash bhi ho toh data safe
 *
 * 3. ANOMALIES — kya galat ho sakta hai:
 *    → Dirty Read          : Uncommitted data padhna
 *    → Non-Repeatable Read : Same data 2 baar padho — alag value mile
 *    → Phantom Read        : Same query 2 baar chalao — alag rows mile
 *
 * 4. ISOLATION LEVELS — kitna strict rehna hai:
 *    → Read Uncommitted : Sabse kam safe (mat use karo)
 *    → Read Committed   : Default, accha balance (PostgreSQL default)
 *    → Repeatable Read  : Zyada safe, reports ke liye
 *    → Serializable     : Sabse safe, banking ke liye
 *
 * 5. PRACTICAL TIPS:
 *    → Multiple updates hain → transaction use karo
 *    → Hamesha errors handle karo
 *    → Transactions short rakho
 *    → Edge cases test karo
 *    → Kabhi open mat chhodo transaction ko
 */